In [ ]:
import os
import importlib
import h5py as h5
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import chi2
from lambda_commissioning import utils, caltools, vistools
from lambda_commissioning.constants import NARRIBRI

importlib.reload(utils)
print(os.getcwd())
verbose = True

# Use cupy for the batched eigendecomposition / einsums if it is available and
# supports batched `linalg.eigh`; otherwise fall back to numpy on the CPU.
try:
    import cupy as xp
    xp.linalg.eigh(xp.eye(2, dtype=xp.complex64)[None])
    GPU = True
except Exception:
    xp = np
    GPU = False
print(f"Backend: {'cupy (GPU)' if GPU else 'numpy (CPU)'}")

def to_np(a):
    return xp.asnumpy(a) if GPU else a

In [ ]:

# Matplotlib settings for a PASA 2-column paper.
# Single-column width 84 mm ≈ 3.31 in; two-column spanning 174 mm ≈ 6.85 in.
import matplotlib as mpl

PASA_COL_W  = 3.31   # single-column width, inches
PASA_2COL_W = 6.85   # two-column spanning width, inches

mpl.rcParams.update({
    'font.family':        'serif',
    'font.size':           8,
    'axes.titlesize':      8,
    'axes.labelsize':      8,
    'xtick.labelsize':     7,
    'ytick.labelsize':     7,
    'legend.fontsize':     7,
    'lines.linewidth':     0.9,
    'axes.linewidth':      0.7,
    'xtick.major.width':   0.7,
    'ytick.major.width':   0.7,
    'figure.dpi':          150,
    'savefig.dpi':         300,
    'savefig.bbox':       'tight',
})

# Scale factor for overlay_sky_sources markers/labels (1.0 = default size).
OVERLAY_SCALE = 0.4


In [ ]:
stokes = "XX"
visibilities_filename = "/data/BLACKMESA_1/sma075/cuda-spatial-filtering/build/apps/visibilities_20260611-1511_172_179_ALVEO0_ALVEO1_ALVEO2_ALVEO3.hdf5"

# If set to a filename, gains are loaded from that HDF5 file instead of
# from visibilities_filename and applied (transferred) to this capture --
# e.g. to apply a calibration solution solved against a different observation.
# Leave as None to use the gains stored alongside this capture's own visibilities.
gain_transfer_filename = "/data/BLACKMESA_1/sma075/cuda-spatial-filtering/build/apps/visibilities_20260611-1349_172_179_ALVEO0_ALVEO1_ALVEO2_ALVEO3.hdf5"


visCorrList, antPairs = utils.read_hdf5_data_capture(visibilities_filename, [stokes], returnCorrMatrix=True)
if gain_transfer_filename is not None:
    if verbose:
        print(f"Transferring gains from {gain_transfer_filename}")
    gains = utils.read_hdf5_gains(gain_transfer_filename, [stokes])
else:
    gains = utils.read_hdf5_gains(visibilities_filename, [stokes])
timeVec, freqs = utils.read_hdf5_time_freq(visibilities_filename, verbose=verbose)

antIDvec = np.unique(antPairs)
Na, Nc, Nt = antIDvec.size, freqs.size, timeVec.gps.size
print(f"Na={Na}, Nc={Nc}, Nt={Nt}")

LAMBDA36 = utils.make_telescope_model(antIDvec, telescope="LAMBDA36", verbose=verbose)
antLocsAll = np.array([LAMBDA36.east, LAMBDA36.north]).T

flag_ants = "8,12"
flagAnts = [int(x.strip()) for x in flag_ants.split(",")]
goodInds = ~np.isin(antIDvec, flagAnts)
antLocs = antLocsAll[goodInds]
Ng = int(goodInds.sum())

# drop first 20 and last 20 samples

print(f"Flagging antenna IDs {flagAnts}, {Ng} antennas remain")

In [ ]:
channel_to_plot = 4
dft_size = 256
meanFreq = float(freqs[channel_to_plot])

BATCH_SIZE = 10000
batch = 6



visTensor = visCorrList[0]  # (Nt, Nc, Na, Na)
visCal = caltools.apply_gains(visTensor, gains[stokes])
tindVec = np.arange(10 + batch * BATCH_SIZE, min((batch + 1) * BATCH_SIZE + 10, visCal.shape[0] - 10))
# Drop flagged antennas entirely (rather than zeroing rows/columns) so that
# each per-timestep covariance matrix is full rank for the eigen-analysis.
R = visCal[tindVec][:, channel_to_plot][:, goodInds][:, :, goodInds]  # (Nt, Ng, Ng)
R = xp.asarray(np.ascontiguousarray(R))
Nt = R.shape[0]
print(f"R shape: {R.shape}, dtype: {R.dtype}")

In [ ]:
channel_to_plot_sun = 4
tindVec_sun = np.arange(10, 15)

# Drop flagged antennas entirely (rather than zeroing rows/columns) so that
# each per-timestep covariance matrix is full rank for the eigen-analysis.
R_sun = visCal[tindVec_sun][:, channel_to_plot_sun][:, goodInds][:, :, goodInds]  # (Nt, Ng, Ng)
R_sun = xp.asarray(np.ascontiguousarray(R_sun))
Nt_sun = R_sun.shape[0]
print(f"R shape: {R_sun.shape}, dtype: {R_sun.dtype}")
import math
# Number of raw correlator samples integrated into each visibility timestep --
# the snapshot count L used by the hypothesis test below.
# Na is rounded up to the nearest multiple of 10 because the correlator pads
# antenna slots to groups of 10.
with h5.File(visibilities_filename, "r") as hf:
    Npkts_per_vis = hf["vis_missing_nums"][0][1]
Na_padded = math.ceil(Na / 10) * 10
Ntime_pkts = Npkts_per_vis / Nc / Na_padded
L = int(Ntime_pkts * 64)
print(f"Na_padded={Na_padded}, Snapshots per integration: L = {L}")

# Whiten by per-antenna noise power (the autocorrelations) so that, under the
# null hypothesis of no correlated signal/RFI, all eigenvalues of Rnorm are
# approximately equal -- the condition the sphericity test below assumes.
autopow = xp.real(xp.diagonal(R_sun, axis1=-2, axis2=-1))  # (Nt, Ng)
Dinv = 1.0 / xp.sqrt(autopow)
Rnorm_sun = R_sun * Dinv[:, :, None] * Dinv[:, None, :]

eigvals, eigvecs = xp.linalg.eigh(Rnorm_sun)
# take the brightest eigenvector as the sun
sun_eigvec = eigvecs[0, :, -1]


In [ ]:
import math
# Number of raw correlator samples integrated into each visibility timestep --
# the snapshot count L used by the hypothesis test below.
# Na is rounded up to the nearest multiple of 10 because the correlator pads
# antenna slots to groups of 10.
with h5.File(visibilities_filename, "r") as hf:
    Npkts_per_vis = hf["vis_missing_nums"][0][1]
Na_padded = math.ceil(Na / 10) * 10
Ntime_pkts = Npkts_per_vis / Nc / Na_padded
L = int(Ntime_pkts * 64)
print(f"Na_padded={Na_padded}, Snapshots per integration: L = {L}")

# Whiten by per-antenna noise power (the autocorrelations) so that, under the
# null hypothesis of no correlated signal/RFI, all eigenvalues of Rnorm are
# approximately equal -- the condition the sphericity test below assumes.
autopow = xp.real(xp.diagonal(R, axis1=-2, axis2=-1))  # (Nt, Ng)
Dinv = 1.0 / xp.sqrt(autopow)
Rnorm = R * Dinv[:, :, None] * Dinv[:, None, :]


In [ ]:
Ntime_pkts

In [ ]:
# Batched eigendecomposition: one (Ng, Ng) eigh per timestep, vectorized.
eigvals, eigvecs = xp.linalg.eigh(Rnorm)  # ascending: (Nt, Ng), (Nt, Ng, Ng)

fig, ax = plt.subplots(figsize=(PASA_COL_W, PASA_COL_W * 0.7), constrained_layout=True)
ax.plot(to_np(eigvals))
ax.set_xlabel("Time Step")
ax.set_ylabel("Eigenvalue")
plt.show()

def estimate_num_signals(eigvals_asc, L, n_sigma=3.0):
    """Robust noise-floor signal counter.

    Assumes signals << N. Estimates noise eigenvalue distribution from
    the 20th-80th percentile range, then counts eigenvalues exceeding
    median + 3*sigma as signals.
    """
    # Robust noise floor stats per timestep
    p20 = xp.percentile(eigvals_asc, 20, axis=1)
    p50 = xp.percentile(eigvals_asc, 50, axis=1)
    p80 = xp.percentile(eigvals_asc, 80, axis=1)

    # 20-80 range spans 2 * 0.8416 * sigma for a normal distribution
    sigma = (p80 - p20) / (2 * 0.8416)

    threshold = p50 + 4.0 * sigma  # (Nt_,)
    print(threshold)
    return np.maximum(xp.sum(eigvals_asc > threshold[:, None], axis=1), 1)

d = estimate_num_signals(eigvals, L)
d_np = to_np(d)
print(f"Signal eigenvector count d: min={d_np.min()}, mean={d_np.mean():.2f}, max={d_np.max()}, L={L}")


In [ ]:

# Attenuation factor applied to the Sun's eigenvalue during reconstruction.
# 1.0 = keep the Sun at full power; 0.0 = remove it entirely.
sun_eigenvalue_scale = 1.0

ranks = xp.arange(Ng)[None, :]
noise_mask  = (ranks < (Ng - d[:, None])).astype(eigvals.dtype)   # (Nt_, Ng)
signal_mask = 1.0 - noise_mask                                      # (Nt_, Ng)

# Project sun_eigvec onto each eigenvector: coordinates in eigvec basis
# eigvecs: (Nt_, Ng, Ng), column j = j-th eigenvector
sun_coords = xp.einsum('tji,j->ti', xp.conj(eigvecs), sun_eigvec)  # (Nt_, Ng)

# Keep only signal-subspace part and normalize → Sun direction in eigvec coords
sun_coords_sig  = sun_coords * signal_mask
sun_coords_norm = sun_coords_sig / xp.linalg.norm(sun_coords_sig, axis=1, keepdims=True)

# Lift Sun direction back to N-space
b = xp.einsum('tij,tj->ti', eigvecs, sun_coords_norm)              # (Nt_, Ng)

# Power in Sun direction = eigenvalue-weighted projection
lambda_sun = xp.einsum('ti,ti->t', eigvals, xp.abs(sun_coords_norm)**2).real  # (Nt_,)
lambda_sun_scaled = sun_eigenvalue_scale * lambda_sun

# Reconstruct: noise subspace + Sun component (d-1 RFI interferers removed)
R_clean_norm = (
    xp.einsum('tij,tj,tkj->tik', eigvecs, noise_mask * eigvals, xp.conj(eigvecs), optimize=True)
    + lambda_sun_scaled[:, None, None] * xp.einsum('ti,tk->tik', b, xp.conj(b))
)

# Noise+Sun projector for Leshem correction (direction kept at full weight)
Pn = (
    xp.einsum('tij,tj,tkj->tik', eigvecs, noise_mask, xp.conj(eigvecs), optimize=True)
    + xp.einsum('ti,tk->tik', b, xp.conj(b))
)


In [ ]:
# Leshem (2000) correction: vec(Pn R Pn) = kron(Pn, conj(Pn)) @ vec(R), so the
# time-average of sum_t kron(Pn_t, conj(Pn_t)) is the linear operator relating
# the (assumed slowly-varying) true covariance to the time-averaged projected
# covariance. Built as a single matmul over the flattened projectors rather
# than Nt explicit (Ng^2 x Ng^2) Kronecker products.
N2 = Ng * Ng
Eflat = Pn.reshape(Nt, N2)
Fflat = xp.conj(Pn).reshape(Nt, N2)
Wavg = (Eflat.T @ Fflat).reshape(Ng, Ng, Ng, Ng).transpose(0, 2, 1, 3).reshape(N2, N2) / Nt
Wavg_np = to_np(Wavg)
print(f"cond(Wavg) = {np.linalg.cond(Wavg_np):.3e}")

vecRcleanAvg = to_np(R_clean_norm.mean(axis=0)).flatten()
rcond = 1e-3  # truncate small singular values of Wavg before inverting.
Wpinv = np.linalg.pinv(Wavg_np, rcond=rcond)
R_corrected_norm = (Wpinv @ vecRcleanAvg).reshape(Ng, Ng)

In [ ]:

# De-whiten back to physical units and form the three comparison images.
D = to_np(xp.sqrt(autopow)).mean(axis=0)  # (Ng,)
calMean = to_np(R).mean(axis=0)
R_clean = to_np(R_clean_norm).mean(axis=0) * D[:, None] * D[None, :]
R_corrected = R_corrected_norm * D[:, None] * D[None, :]

skyOrig = vistools.DFT_image(calMean, dft_size, antLocs, meanFreq)
skyMitigated = vistools.DFT_image(R_clean, dft_size, antLocs, meanFreq)
skyLeshem = vistools.DFT_image(R_corrected, dft_size, antLocs, meanFreq)

# 3 rows × 2 cols of square panels: height ≈ 1.5 × width
fig, axs = plt.subplots(3, 2, figsize=(PASA_2COL_W, PASA_2COL_W * 1.25), constrained_layout=True)
t_mid = timeVec[tindVec[len(tindVec) // 2]]
for ax, img, title in zip(
    axs.flat,
    [np.abs(skyOrig).T, np.abs(skyMitigated).T, np.abs(skyLeshem).T,
     np.abs(skyLeshem).T - np.abs(skyMitigated).T,
     np.abs(skyMitigated).T - np.abs(skyOrig).T,
     np.abs(skyLeshem).T - np.abs(skyOrig).T],
    ["Original", "Eigenfilter", "Eigenfilter + Leshem",
     "Leshem $-$ Eigenfilter", "Eigenfilter $-$ Original", "Leshem $-$ Original"],
):
    im = ax.imshow(img, extent=[-1, 1, -1, 1], origin="lower")
    fig.colorbar(im, ax=ax, label="Intensity", pad=0.02, shrink=0.8)
    vistools.overlay_sky_sources(ax, t_mid, NARRIBRI, scale=OVERLAY_SCALE)
    ax.set_xlabel("$l$")
    ax.set_ylabel("$m$")
    ax.set_title(f"{title}  ({stokes}, {meanFreq/1e6:.1f} MHz)")
axs.flat[-1].legend(loc='upper right', fontsize=6)
plt.show()


In [ ]:

# --- Sun peeling (per-timestep rank-1 model) ------------------------------
# Each timestep's Sun contribution is the rank-1 term lambda_sun_scaled(t) * b(t) b(t)^H
# isolated in the projection step. Subtracting it removes the Sun + all sidelobes exactly;
# we then restore a clean (sidelobe-free) Gaussian at the known Sun position.
from astropy.coordinates import get_sun, AltAz
from lambda_commissioning.constants import NARRIBRI
from lambda_commissioning.modelling import calc_lmn

def sun_lm(t):
    altaz = get_sun(t).transform_to(AltAz(obstime=t, location=NARRIBRI))
    l, m, _ = calc_lmn(altaz.alt.deg, altaz.az.deg, degrees=True)
    return float(l), float(m)

l_sun, m_sun = sun_lm(timeVec[tindVec[len(tindVec) // 2]])
clean_sigma = 0.04   # clean-beam size in l/m -- match the dirty-beam main lobe
sun_clip_percentile = 99.0  # clip restored image at this percentile to reveal background

# Peel the Sun down to the noise floor (not to zero) so the restored image
# starts from a physically meaningful baseline rather than a hard subtraction.
lambda_noise = float(to_np(xp.median(eigvals, axis=1).mean()))
lambda_sun_mean = float(to_np(lambda_sun_scaled.mean()))
sun_peel_frac = max(0.0, 1.0 - lambda_noise / lambda_sun_mean) if lambda_sun_mean > 0 else 0.0
print(f"Sun peel fraction: {sun_peel_frac:.3f}  (lambda_sun={lambda_sun_mean:.3f}, lambda_noise={lambda_noise:.3f})")

# Time-averaged rank-1 Sun model (using the scaled eigenvalue) and the Sun-free residual.
R_sun_norm = (lambda_sun_scaled[:, None, None] *
              xp.einsum('ti,tk->tik', b, xp.conj(b))).mean(axis=0)
R_resid = to_np(R_clean_norm.mean(axis=0) - sun_peel_frac * R_sun_norm) * D[:, None] * D[None, :]
R_sun   = to_np(R_sun_norm) * D[:, None] * D[None, :]

skySun   = vistools.DFT_image(R_sun,   dft_size, antLocs, meanFreq)  # Sun + sidelobes
skyResid = vistools.DFT_image(R_resid, dft_size, antLocs, meanFreq)  # Sun peeled away

_lvec = np.linspace(-1, 1, dft_size)
Lg, Mg = np.meshgrid(_lvec, _lvec, indexing="ij")  # native DFT_image [il, im] layout
disk_mask = (Lg**2 + Mg**2) < 1.0
clean_sun = np.abs(skySun).max() * np.exp(
    -((Lg - l_sun) ** 2 + (Mg - m_sun) ** 2) / (2 * clean_sigma ** 2))
clean_sun[~disk_mask] = 0.0                          # zero outside the horizon
skyRestored = np.abs(skyResid) + clean_sun       # real, native [il, im] layout

# Clip so the bright Sun peak does not saturate the background.
skyRestored_clipped = np.clip(skyRestored, 0, np.percentile(skyRestored, sun_clip_percentile))

# 3 panels stacked vertically: each is one column wide, aspect ratio is square.
fig, axs = plt.subplots(3, 1, figsize=(PASA_COL_W, PASA_COL_W * 2.5), constrained_layout=True)
for ax, img, title in zip(
    axs,
    [np.log10(np.abs(skySun).T), np.log10(np.abs(skyResid).T), np.log10(skyRestored_clipped.T)],
    ["Sun model (dirty)", "Sun peeled", "Restored (clipped)"],
):
    im = ax.imshow(img, extent=[-1, 1, -1, 1], origin="lower")
    fig.colorbar(im, ax=ax, label="Intensity", pad=0.02, shrink=0.8)
    vistools.overlay_sky_sources(ax, timeVec[tindVec[len(tindVec) // 2]], NARRIBRI, scale=OVERLAY_SCALE)
    ax.set_xlabel("$l$")
    ax.set_ylabel("$m$")
    ax.set_title(f"{title}  ({stokes}, {meanFreq/1e6:.1f} MHz)")
plt.show()


In [ ]:
# --- Source SNR utilities ------------------------------------------------
# Measure the peak signal-to-noise of a point source (the Sun) in a DFT dirty
# image. Noise is taken from a quiet patch of the image so scattered artefacts
# / sidelobes don't inflate it: either auto-pick the lowest-MAD square, or pass
# an explicit `noise_region`.
import matplotlib.patches as mpatches


def _find_quiet_box(img, L, M, disk, src_mask, box=0.25, step=0.08):
    """Slide a square window over the in-disk, off-source sky and return the
    box (lmin, lmax, mmin, mmax) with the lowest robust spread (MAD)."""
    hw = box / 2.0
    cls = np.arange(L.min() + hw, L.max() - hw + 1e-9, step)
    cms = np.arange(M.min() + hw, M.max() - hw + 1e-9, step)
    best = None
    for cm in cms:
        for cl in cls:
            box_mask = (np.abs(L - cl) <= hw) & (np.abs(M - cm) <= hw)
            if box_mask.sum() == 0 or (box_mask & ~disk).any() or (box_mask & src_mask).any():
                continue
            pix = img[box_mask]
            spread = np.median(np.abs(pix - np.median(pix)))
            if best is None or spread < best[0]:
                best = (spread, (cl - hw, cl + hw, cm - hw, cm + hw))
    if best is None:
        raise ValueError("No quiet box fits inside the disk; reduce `box`.")
    return best[1]


def measure_source_snr(sky, source_lm=None, src_radius=0.08, noise_region=None,
                       noise_box=0.25, extent=(-1, 1, -1, 1), robust=True,
                       return_details=False):
    """Peak signal-to-noise ratio of a point source in a DFT dirty image.

    Signal = peak |image| within `src_radius` of the source, minus the local
    background. Noise = robust spread of |image| over a quiet region:
        noise_region=None              -> auto-pick the quietest noise_box square
        noise_region=("box", l0,l1,m0,m1)
        noise_region=("circle", lc,mc,r)
    """
    img = np.abs(sky).T  # match displayed orientation: l -> x, m -> y
    n = img.shape[0]
    l = np.linspace(extent[0], extent[1], n)
    m = np.linspace(extent[2], extent[3], n)
    L, M = np.meshgrid(l, m)
    rfield = np.hypot(L, M)
    disk = rfield <= 1.0

    if source_lm is None:
        masked = np.where(disk, img, -np.inf)
        iy, ix = np.unravel_index(np.argmax(masked), img.shape)
        l0, m0 = L[iy, ix], M[iy, ix]
    else:
        l0, m0 = source_lm
    src_mask = np.hypot(L - l0, M - m0) <= src_radius
    peak = img[src_mask].max()

    if noise_region is None:
        noise_region = ("box", *_find_quiet_box(img, L, M, disk, src_mask, box=noise_box))

    if noise_region[0] == "box":
        _, l_lo, l_hi, m_lo, m_hi = noise_region
        noise_mask = (L >= l_lo) & (L <= l_hi) & (M >= m_lo) & (M <= m_hi)
    elif noise_region[0] == "circle":
        _, lc, mc, r = noise_region
        noise_mask = np.hypot(L - lc, M - mc) <= r
    else:
        raise ValueError(f"Unknown noise_region kind: {noise_region[0]!r}")
    noise_mask &= disk & ~src_mask
    bg_pix = img[noise_mask]

    if robust:
        background = np.median(bg_pix)
        noise = 1.4826 * np.median(np.abs(bg_pix - background))
    else:
        background = bg_pix.mean()
        noise = bg_pix.std()
    snr = (peak - background) / noise

    if return_details:
        return snr, {"peak": peak, "background": background, "noise": noise,
                     "source_lm": (l0, m0), "noise_region": noise_region,
                     "n_noise_pix": int(noise_mask.sum())}
    return snr


def plot_source_snr(sky, ax=None, extent=(-1, 1, -1, 1), **kwargs):
    """Image the field, overlay the source aperture (red) and noise region
    (cyan). Returns (snr, info, ax)."""
    snr, info = measure_source_snr(sky, extent=extent, return_details=True, **kwargs)
    l0, m0 = info["source_lm"]
    src_radius = kwargs.get("src_radius", 0.08)
    if ax is None:
        _, ax = plt.subplots(figsize=(PASA_COL_W, PASA_COL_W), constrained_layout=True)
    im = ax.imshow(np.abs(sky).T, extent=extent, origin="lower")
    ax.figure.colorbar(im, ax=ax, label="Intensity", shrink=0.8)
    ax.add_patch(mpatches.Circle((l0, m0), src_radius, fill=False, edgecolor="r", lw=1.5))
    ax.plot(l0, m0, "r+", ms=10)
    nr = info["noise_region"]
    if nr[0] == "box":
        _, l_lo, l_hi, m_lo, m_hi = nr
        ax.add_patch(mpatches.Rectangle((l_lo, m_lo), l_hi - l_lo, m_hi - m_lo,
                                        fill=False, edgecolor="c", lw=1.5))
    else:
        _, lc, mc, r = nr
        ax.add_patch(mpatches.Circle((lc, mc), r, fill=False, edgecolor="c", lw=1.5))
    ax.set_xlabel("$l$"); ax.set_ylabel("$m$")
    ax.set_title(f"SNR = {snr:.1f}  (noise from {nr[0]})")
    return snr, info, ax


In [ ]:
# Sanity-check the SNR measurement and the auto-chosen quiet noise box.
snr, info, ax = plot_source_snr(skyOrig, src_radius=0.08, noise_box=0.25)
plt.show()
print(f"SNR={snr:.1f}, noise={info['noise']:.3e}, region={info['noise_region']}")

In [ ]:

# ── Full-sample static comparison images ─────────────────────────────────────
# Re-process every batch and accumulate time-averaged covariance matrices for
# five cases: (1) RFI-affected original, (2) adjacent clean channel (ch5),
# (3) second clean reference channel, (4) eigenfilter no-Leshem, (5) eigenfilter + Leshem.
from tqdm import tqdm

_clean_channel = 5
_clean_freq = float(freqs[_clean_channel])

_clean_channel2 = 2          # second clean reference channel (no RFI)
_clean_freq2 = float(freqs[_clean_channel2])


def _count_signals_fs(eigvals_asc, n_sigma=5.0):
    p20 = xp.percentile(eigvals_asc, 20, axis=1)
    p50 = xp.percentile(eigvals_asc, 50, axis=1)
    p80 = xp.percentile(eigvals_asc, 80, axis=1)
    sigma = (p80 - p20) / (2 * 0.8416)
    return np.maximum(xp.sum(eigvals_asc > (p50 + n_sigma * sigma)[:, None], axis=1), 1)


FS_BATCH = 5000
_total_t = visCal.shape[0] - 20
_n_batches = _total_t // FS_BATCH
_N2 = Ng * Ng

_R_rfi = np.zeros((Ng, Ng), dtype=complex)
_R_adj = np.zeros((Ng, Ng), dtype=complex)
_R_ch2 = np.zeros((Ng, Ng), dtype=complex)
_R_nol = np.zeros((Ng, Ng), dtype=complex)
_R_les = np.zeros((Ng, Ng), dtype=complex)
_total_w = 0

for _b in tqdm(range(_n_batches), desc="Full-sample pass"):
    _tind = np.arange(10 + _b * FS_BATCH, 10 + (_b + 1) * FS_BATCH)
    _w = len(_tind)

    _Rb = np.ascontiguousarray(visCal[_tind][:, channel_to_plot][:, goodInds][:, :, goodInds])
    _R_rfi += _Rb.mean(axis=0) * _w

    _Rc = np.ascontiguousarray(visCal[_tind][:, _clean_channel][:, goodInds][:, :, goodInds])
    _R_adj += _Rc.mean(axis=0) * _w

    _Rc2 = np.ascontiguousarray(visCal[_tind][:, _clean_channel2][:, goodInds][:, :, goodInds])
    _R_ch2 += _Rc2.mean(axis=0) * _w

    # Eigendecomposition
    _Rb_xp = xp.asarray(_Rb)
    _ap = xp.real(xp.diagonal(_Rb_xp, axis1=-2, axis2=-1))
    _Di = 1.0 / xp.sqrt(_ap)
    _Rn = _Rb_xp * _Di[:, :, None] * _Di[:, None, :]
    _ev, _evec = xp.linalg.eigh(_Rn)
    _db = _count_signals_fs(_ev)

    _rk = xp.arange(Ng)[None, :]
    _nm = (_rk < (Ng - _db[:, None])).astype(_ev.dtype)
    _sm = 1.0 - _nm

    _sc = xp.einsum('tji,j->ti', xp.conj(_evec), sun_eigvec)
    _sc_sig = _sc * _sm
    _sc_norm = _sc_sig / xp.linalg.norm(_sc_sig, axis=1, keepdims=True)
    _bv = xp.einsum('tij,tj->ti', _evec, _sc_norm)
    _ls = xp.einsum('ti,ti->t', _ev, xp.abs(_sc_norm) ** 2).real
    _ls_sc = sun_eigenvalue_scale * _ls

    _Rcn = (
        xp.einsum('tij,tj,tkj->tik', _evec, _nm * _ev, xp.conj(_evec), optimize=True)
        + _ls_sc[:, None, None] * xp.einsum('ti,tk->tik', _bv, xp.conj(_bv))
    )
    _Pn = (
        xp.einsum('tij,tj,tkj->tik', _evec, _nm, xp.conj(_evec), optimize=True)
        + xp.einsum('ti,tk->tik', _bv, xp.conj(_bv))
    )
    _Db = to_np(xp.sqrt(_ap)).mean(axis=0)

    # No-Leshem
    _R_nol += to_np(_Rcn.mean(axis=0)) * _Db[:, None] * _Db[None, :] * _w

    # Leshem correction
    _Ef = _Pn.reshape(_w, _N2)
    _Ff = xp.conj(_Pn).reshape(_w, _N2)
    _Wa = (_Ef.T @ _Ff).reshape(Ng, Ng, Ng, Ng).transpose(0, 2, 1, 3).reshape(_N2, _N2) / _w
    _vR = to_np(_Rcn.mean(axis=0)).flatten()
    _Wp = np.linalg.pinv(to_np(_Wa), rcond=1e-3)
    _R_les += (_Wp @ _vR).reshape(Ng, Ng) * _Db[:, None] * _Db[None, :] * _w

    _total_w += _w

sky_rfi_full  = vistools.DFT_image(_R_rfi / _total_w, dft_size, antLocs, meanFreq)
sky_adj_full  = vistools.DFT_image(_R_adj / _total_w, dft_size, antLocs, _clean_freq)
sky_ch2_full  = vistools.DFT_image(_R_ch2 / _total_w, dft_size, antLocs, _clean_freq2)
sky_nol_full  = vistools.DFT_image(_R_nol / _total_w, dft_size, antLocs, meanFreq)
sky_les_full  = vistools.DFT_image(_R_les / _total_w, dft_size, antLocs, meanFreq)

print(f"Full sample: {_total_w} timesteps, {_n_batches} batches.")


In [ ]:

# ── 4-panel comparison figure + LaTeX SNR table (full sample) ─────────────────
_t_mid_fs = timeVec[10 + _total_t // 2]
_sun_pos_fs = sun_lm(_t_mid_fs)

_snr_rfi = measure_source_snr(sky_rfi_full, source_lm=_sun_pos_fs, src_radius=0.08)
_snr_adj = measure_source_snr(sky_adj_full, source_lm=_sun_pos_fs, src_radius=0.08)
_snr_nol = measure_source_snr(sky_nol_full, source_lm=_sun_pos_fs, src_radius=0.08)
_snr_les = measure_source_snr(sky_les_full, source_lm=_sun_pos_fs, src_radius=0.08)

_panels = [
    (sky_rfi_full, f"RFI-affected ({meanFreq/1e6:.1f} MHz)",   _snr_rfi),
    (sky_adj_full, f"Adjacent clean ({_clean_freq/1e6:.1f} MHz)", _snr_adj),
    (sky_nol_full, "Eigenfilter (no Leshem)",                    _snr_nol),
    (sky_les_full, "Eigenfilter + Leshem",                       _snr_les),
]

_vmin = min(np.abs(sky).min() for sky, _, _ in _panels)
_vmax = max(np.abs(sky).max() for sky, _, _ in _panels)

# 2×2 grid; height reduced to match two rows of square panels after colorbar space.
fig, axs = plt.subplots(2, 2, figsize=(PASA_2COL_W, PASA_2COL_W * 0.85), constrained_layout=True)
fig.get_layout_engine().set(hspace=0.05)
for ax, (sky, title, snr) in zip(axs.flat, _panels):
    im = ax.imshow(np.abs(sky).T, extent=[-1, 1, -1, 1], origin="lower",
                   vmin=_vmin, vmax=_vmax)
    vistools.overlay_sky_sources(ax, _t_mid_fs, NARRIBRI, scale=OVERLAY_SCALE)
    ax.set_xlabel("$l$")
    ax.set_ylabel("$m$")
    ax.set_title(f"{title}\n{stokes}, SNR = {snr:.1f}")
fig.colorbar(im, ax=axs, label="Intensity", pad=0.02, shrink=0.9)
plt.savefig("full_sample_comparison.pdf")
plt.show()

# ── LaTeX SNR table ───────────────────────────────────────────────────────────
_rows = [
    ("RFI-affected",           f"{meanFreq/1e6:.1f}",  _snr_rfi),
    ("Adjacent clean channel", f"{_clean_freq/1e6:.1f}", _snr_adj),
    ("Eigenfilter (no Leshem)",f"{meanFreq/1e6:.1f}",  _snr_nol),
    ("Eigenfilter + Leshem",   f"{meanFreq/1e6:.1f}",  _snr_les),
]
print(r"\begin{table}")
print(r"  \centering")
print(r"  \caption{Sun peak SNR for each RFI mitigation case (full sample).}")
print(r"  \label{tab:sun-snr}")
print(r"  \begin{tabular}{lcc}")
print(r"    \hline")
print(r"    Method & Frequency (MHz) & Sun SNR \\")
print(r"    \hline")
for _method, _freq, _snr in _rows:
    print(f"    {_method} & {_freq} & {_snr:.1f} \\\\")
print(r"    \hline")
print(r"  \end{tabular}")
print(r"\end{table}")


In [ ]:

# ── Channel-combination SNR comparison ───────────────────────────────────────
# Question: does eigenfilter-corrected ch4 contribute the same SNR gain as a
# second genuine clean channel when averaged with the clean reference channel?
#
# Theory: averaging N independent equal-noise channels improves SNR by sqrt(N).
# So clean-ref alone → clean-ref + adj-clean (two clean) gives √2 ≈ 1.414× gain.
# clean-ref + eigenfilter(RFI) should match this if the correction is lossless.
# clean-ref + raw-RFI is the baseline showing what happens without any correction.
#
# Combination is done in visibility (covariance matrix) space — average the two
# matrices and image at the mean frequency.  This is the optimal combination
# under equal noise across channels.

_mean_freq_2clean    = (_clean_freq  + _clean_freq2) / 2.0
_mean_freq_ref_rfi   = (_clean_freq2 + meanFreq) / 2.0

# Normalised per-timestep covariance averages (each already weighted by _total_w)
_R_2clean      = (_R_adj + _R_ch2) / (2 * _total_w)   # adj-clean + clean-ref
_R_ref_raw_rfi = (_R_ch2 + _R_rfi) / (2 * _total_w)   # clean-ref + raw RFI (no correction)
_R_ref_nol     = (_R_ch2 + _R_nol) / (2 * _total_w)   # clean-ref + eigenfilter (no Leshem)
_R_ref_les     = (_R_ch2 + _R_les) / (2 * _total_w)   # clean-ref + eigenfilter + Leshem
_R_ref_alone   = _R_ch2 / _total_w                     # clean-ref single channel

sky_2clean      = vistools.DFT_image(_R_2clean,      dft_size, antLocs, _mean_freq_2clean)
sky_ref_raw_rfi = vistools.DFT_image(_R_ref_raw_rfi, dft_size, antLocs, _mean_freq_ref_rfi)
sky_ref_nol     = vistools.DFT_image(_R_ref_nol,     dft_size, antLocs, _mean_freq_ref_rfi)
sky_ref_les     = vistools.DFT_image(_R_ref_les,     dft_size, antLocs, _mean_freq_ref_rfi)

# Measure SNR with a fixed noise region locked from the single clean-ref image.
_snr_ref, _info_ref = measure_source_snr(sky_ch2_full, source_lm=_sun_pos_fs,
                                          src_radius=0.08, return_details=True)
_nr_combo = _info_ref["noise_region"]

_snr_2clean      = measure_source_snr(sky_2clean,      source_lm=_sun_pos_fs, src_radius=0.08,
                                       noise_region=_nr_combo)
_snr_ref_raw_rfi = measure_source_snr(sky_ref_raw_rfi, source_lm=_sun_pos_fs, src_radius=0.08,
                                       noise_region=_nr_combo)
_snr_ref_nol     = measure_source_snr(sky_ref_nol,     source_lm=_sun_pos_fs, src_radius=0.08,
                                       noise_region=_nr_combo)
_snr_ref_les     = measure_source_snr(sky_ref_les,     source_lm=_sun_pos_fs, src_radius=0.08,
                                       noise_region=_nr_combo)

# √2 benchmark: SNR a second clean channel would contribute if it had equal noise.
_sqrt2_expected = np.sqrt(2) * _snr_ref

print("Channel-combination SNR summary")
print(f"  Clean ref alone           : SNR = {_snr_ref:.1f}  (reference)")
print(f"  2 clean channels          : SNR = {_snr_2clean:.1f}  "
      f"({_snr_2clean/_snr_ref:.3f}×,  expect √2={np.sqrt(2):.3f}×)")
print(f"  Clean ref + raw RFI       : SNR = {_snr_ref_raw_rfi:.1f}  "
      f"({_snr_ref_raw_rfi/_snr_2clean*100:.1f}% of 2-clean benchmark)")
print(f"  Clean ref + EF (no Leshem): SNR = {_snr_ref_nol:.1f}  "
      f"({_snr_ref_nol/_snr_2clean*100:.1f}% of 2-clean benchmark)")
print(f"  Clean ref + EF (+Leshem)  : SNR = {_snr_ref_les:.1f}  "
      f"({_snr_ref_les/_snr_2clean*100:.1f}% of 2-clean benchmark)")

# ── Bar chart ─────────────────────────────────────────────────────────────────
_labels = [
    "Clean ref\nalone",
    "2 clean\nchannels",
    "Clean ref\n+ raw RFI",
    "Clean ref\n+ EF (no Leshem)",
    "Clean ref\n+ EF (+Leshem)",
]
_snrs   = [_snr_ref, _snr_2clean, _snr_ref_raw_rfi, _snr_ref_nol, _snr_ref_les]
_colors = ["#4878CF", "#6ACC65", "#D65F5F", "#E08060", "#B47CC7"]

fig, ax = plt.subplots(figsize=(PASA_2COL_W * 0.7, PASA_COL_W), constrained_layout=True)
bars = ax.bar(_labels, _snrs, color=_colors, width=0.55)
ax.axhline(_sqrt2_expected, color="k", ls="--", lw=0.9,
           label=f"$\\sqrt{{2}}\\times$ clean ref = {_sqrt2_expected:.1f}")
ax.axhline(_snr_ref, color="grey", ls=":", lw=0.9,
           label=f"Clean ref alone = {_snr_ref:.1f}")
for bar, snr in zip(bars, _snrs):
    ax.text(bar.get_x() + bar.get_width() / 2, snr + 0.5, f"{snr:.1f}",
            ha="center", va="bottom", fontsize=6.5)
ax.set_ylabel("Sun peak SNR")
ax.set_title(f"SNR gain from channel combination  ({stokes})")
ax.legend(fontsize=6)
ax.set_ylim(0, max(_snrs) * 1.18)
plt.savefig("channel_combination_snr.pdf")
plt.show()

# ── 4-panel dirty images ──────────────────────────────────────────────────────
_combo_panels = [
    (sky_ch2_full,    "Clean ref alone",               _snr_ref),
    (sky_2clean,      "2 clean channels",              _snr_2clean),
    (sky_ref_raw_rfi, "Clean ref + raw RFI",           _snr_ref_raw_rfi),
    (sky_ref_les,     "Clean ref + EF (+Leshem)",      _snr_ref_les),
]
fig2, axs2 = plt.subplots(1, 4, figsize=(PASA_2COL_W, PASA_2COL_W / 4 * 1.3),
                           constrained_layout=True)
_cvmin = min(np.abs(sky).min() for sky, _, _ in _combo_panels)
_cvmax = max(np.abs(sky).max() for sky, _, _ in _combo_panels)
for ax, (sky, title, snr) in zip(axs2, _combo_panels):
    im2 = ax.imshow(np.abs(sky).T, extent=[-1, 1, -1, 1], origin="lower",
                    vmin=_cvmin, vmax=_cvmax)
    vistools.overlay_sky_sources(ax, _t_mid_fs, NARRIBRI, scale=OVERLAY_SCALE)
    ax.set_xlabel("$l$");  ax.set_ylabel("$m$")
    pct = snr / _snr_2clean * 100
    ax.set_title(f"{title}\nSNR = {snr:.1f}  ({pct:.0f}% of 2-clean)", fontsize=7)
fig2.colorbar(im2, ax=axs2, label="Intensity", pad=0.02, shrink=0.8)
plt.savefig("channel_combination_images.pdf")
plt.show()

# ── LaTeX table ───────────────────────────────────────────────────────────────
print(r"\begin{table}")
print(r"  \centering")
print(r"  \caption{Sun peak SNR for channel combinations. The $\sqrt{2}$ benchmark")
print(f"           is {_sqrt2_expected:.1f} (= $\\sqrt{{2}}\\times${_snr_ref:.1f}).}}")
print(r"  \label{tab:combo-snr}")
print(r"  \begin{tabular}{lcc}")
print(r"    \hline")
print(r"    Combination & Sun SNR & \% of $\sqrt{2}$ benchmark \\")
print(r"    \hline")
_combo_rows = [
    ("Clean ref alone",            _snr_ref,         _snr_ref/_snr_2clean*100),
    ("2 clean channels",           _snr_2clean,       100.0),
    ("Clean ref + raw RFI",        _snr_ref_raw_rfi,  _snr_ref_raw_rfi/_snr_2clean*100),
    ("Clean ref + EF (no Leshem)", _snr_ref_nol,      _snr_ref_nol/_snr_2clean*100),
    ("Clean ref + EF (+Leshem)",   _snr_ref_les,      _snr_ref_les/_snr_2clean*100),
]
for _method, _snr, _pct in _combo_rows:
    print(f"    {_method} & {_snr:.1f} & {_pct:.1f}\\% \\\\")
print(r"    \hline")
print(r"  \end{tabular}")
print(r"\end{table}")


In [ ]:
from tqdm import tqdm
from matplotlib.animation import FFMpegWriter
import matplotlib.patches as mpatches
from astropy.coordinates import get_sun, AltAz
from lambda_commissioning.constants import NARRIBRI
from lambda_commissioning.modelling import calc_lmn

movie_channel = channel_to_plot      # RFI-affected channel that gets cleaned
clean_channel = 5                    # adjacent RFI-free channel (no cleaning)
movie_freq = float(freqs[movie_channel])
clean_freq = float(freqs[clean_channel])
dft_size_movie = 512
MOVIE_BATCH_SIZE = 5000
src_radius = 0.08
clean_sigma = 0.04                   # restored Sun clean-beam size in l/m
num_batches = (visCal.shape[0] - 20) // MOVIE_BATCH_SIZE

# Pixel grid in DFT_image's native [il, im] layout, for the restored clean Sun.
_lvec = np.linspace(-1, 1, dft_size_movie)
Lgrid, Mgrid = np.meshgrid(_lvec, _lvec, indexing="ij")
disk_mask_movie = (Lgrid**2 + Mgrid**2) < 1.0  # horizon mask for the movie grid

def sun_lm(t):
    """Sun direction cosines (l, m) at time t, from the observation geometry."""
    altaz = get_sun(t).transform_to(AltAz(obstime=t, location=NARRIBRI))
    l, m, _ = calc_lmn(altaz.alt.deg, altaz.az.deg, degrees=True)
    return float(l), float(m)

# Quiet signal-counter (same logic as estimate_num_signals but no debug print).
def _count_signals(eigvals_asc, n_sigma=5.0):
    p20 = xp.percentile(eigvals_asc, 20, axis=1)
    p50 = xp.percentile(eigvals_asc, 50, axis=1)
    p80 = xp.percentile(eigvals_asc, 80, axis=1)
    sigma = (p80 - p20) / (2 * 0.8416)
    return np.maximum(xp.sum(eigvals_asc > (p50 + n_sigma * sigma)[:, None], axis=1), 1)

base_titles = [
    f"Original  ch{movie_channel} ({movie_freq/1e6:.1f} MHz)",
    f"Eigenfilter + Leshem  ch{movie_channel}",
    f"Eigenfilter + No Leshem  ch{movie_channel}",
    f"Adjacent clean  ch{clean_channel} ({clean_freq/1e6:.1f} MHz)",
    f"Sun peeled + restored  ch{movie_channel}",
]
fig, axs2d = plt.subplots(2, 3, figsize=(20, 11), constrained_layout=True)
axs = list(axs2d.flat)
image_axes = axs[:5]
axs[5].axis("off")   # null frame (6th cell of the 3x2 grid)
ims, src_circles, noise_rects = [], [], []
for ax in image_axes:
    ims.append(ax.imshow(np.zeros((dft_size_movie, dft_size_movie)),
                         extent=[-1, 1, -1, 1], origin="lower"))
    circ = mpatches.Circle((0, 0), src_radius, fill=False, edgecolor="r", lw=1.5)
    ax.add_patch(circ)
    src_circles.append(circ)
    # Noise box placeholder -- bounds set from the first frame's pane 0.
    rect = mpatches.Rectangle((0, 0), 0, 0, fill=False, edgecolor="yellow", lw=1.5)
    ax.add_patch(rect)
    noise_rects.append(rect)
    ax.set_xlabel("l")
    ax.set_ylabel("m")

N2 = Ng * Ng
noise_region = None   # locked from first frame, persisted for all subsequent frames
sky_overlay_artists = []  # removed and redrawn each frame on every pane
writer = FFMpegWriter(fps=2)
with writer.saving(fig, "eigenfilter-movie.mp4", dpi=100):
    for b in tqdm(range(num_batches)):
        tind = np.arange(10 + b * MOVIE_BATCH_SIZE, 10 + (b + 1) * MOVIE_BATCH_SIZE)
        sun_pos = sun_lm(timeVec[tind[len(tind) // 2]])

        Rb = xp.asarray(np.ascontiguousarray(
            visCal[tind][:, movie_channel][:, goodInds][:, :, goodInds]
        ))
        Ntb = Rb.shape[0]

        apb = xp.real(xp.diagonal(Rb, axis1=-2, axis2=-1))
        Dinvb = 1.0 / xp.sqrt(apb)
        Rnormb = Rb * Dinvb[:, :, None] * Dinvb[:, None, :]

        evb, evecb = xp.linalg.eigh(Rnormb)
        db = _count_signals(evb)

        ranksb = xp.arange(Ng)[None, :]
        noise_maskb = (ranksb < (Ng - db[:, None])).astype(evb.dtype)
        signal_maskb = 1.0 - noise_maskb

        sun_coordsb = xp.einsum('tji,j->ti', xp.conj(evecb), sun_eigvec)
        sun_coords_sigb = sun_coordsb * signal_maskb
        sun_coords_normb = sun_coords_sigb / xp.linalg.norm(sun_coords_sigb, axis=1, keepdims=True)
        bb = xp.einsum('tij,tj->ti', evecb, sun_coords_normb)
        lambda_sunb = xp.einsum('ti,ti->t', evb, xp.abs(sun_coords_normb) ** 2).real

        R_clean_normb = (
            xp.einsum('tij,tj,tkj->tik', evecb, noise_maskb * evb, xp.conj(evecb), optimize=True)
            + lambda_sunb[:, None, None] * xp.einsum('ti,tk->tik', bb, xp.conj(bb))
        )
        Pnb = (
            xp.einsum('tij,tj,tkj->tik', evecb, noise_maskb, xp.conj(evecb), optimize=True)
            + xp.einsum('ti,tk->tik', bb, xp.conj(bb))
        )

        Eflatb = Pnb.reshape(Ntb, N2)
        Fflatb = xp.conj(Pnb).reshape(Ntb, N2)
        Wavgb = (Eflatb.T @ Fflatb).reshape(Ng, Ng, Ng, Ng).transpose(0, 2, 1, 3).reshape(N2, N2) / Ntb
        vecRb = to_np(R_clean_normb.mean(axis=0)).flatten()
        Wpinvb = np.linalg.pinv(to_np(Wavgb), rcond=1e-3)
        R_correctedb = (Wpinvb @ vecRb).reshape(Ng, Ng)

        Db = to_np(xp.sqrt(apb)).mean(axis=0)
        R_correctedb = R_correctedb * Db[:, None] * Db[None, :]
        calMeanb = to_np(Rb).mean(axis=0)

        # Adjacent RFI-free channel: just time-average and image, no cleaning.
        cleanMeanb = np.ascontiguousarray(
            to_np(xp.asarray(visCal[tind][:, clean_channel][:, goodInds][:, :, goodInds]))
        ).mean(axis=0)

        # Peel Sun to the noise floor (not to zero): compute per-batch peel fraction.
        lambda_noise_b = float(to_np(xp.median(evb, axis=1).mean()))
        lambda_sun_b_mean = float(to_np(lambda_sunb.mean()))
        sun_peel_frac_b = max(0.0, 1.0 - lambda_noise_b / lambda_sun_b_mean) if lambda_sun_b_mean > 0 else 0.0

        R_sun_normb = (lambda_sunb[:, None, None] *
                       xp.einsum('ti,tk->tik', bb, xp.conj(bb))).mean(axis=0)
        R_residb = to_np(R_clean_normb.mean(axis=0) - sun_peel_frac_b * R_sun_normb) * Db[:, None] * Db[None, :]
        R_sunb   = to_np(R_sun_normb) * Db[:, None] * Db[None, :]

        skyOrigb = vistools.DFT_image(calMeanb, dft_size_movie, antLocs, movie_freq)
        skyLeshemb = vistools.DFT_image(R_correctedb, dft_size_movie, antLocs, movie_freq)
        skyNoLeshemb = vistools.DFT_image(
            (R_clean_normb * Db[:, None] * Db[None, :]).mean(axis=0),
            dft_size_movie, antLocs, movie_freq)
        skyCleanb = vistools.DFT_image(cleanMeanb, dft_size_movie, antLocs, clean_freq)
        skyResidb = vistools.DFT_image(R_residb, dft_size_movie, antLocs, movie_freq)
        skySunb   = vistools.DFT_image(R_sunb,   dft_size_movie, antLocs, movie_freq)
        clean_sunb = np.abs(skySunb).max() * np.exp(
            -((Lgrid - sun_pos[0]) ** 2 + (Mgrid - sun_pos[1]) ** 2) / (2 * clean_sigma ** 2))
        clean_sunb[~disk_mask_movie] = 0.0               # zero outside the horizon
        skyRestoredb = np.abs(skyResidb) + clean_sunb   # real, native [il, im] layout

        # Lock the noise region from pane 0 of the very first frame.
        if noise_region is None:
            _, info0 = measure_source_snr(skyOrigb, source_lm=sun_pos,
                                          src_radius=src_radius, return_details=True)
            noise_region = info0["noise_region"]
            _, l_lo, l_hi, m_lo, m_hi = noise_region
            print(f"Noise region locked: l=[{l_lo:.2f},{l_hi:.2f}] m=[{m_lo:.2f},{m_hi:.2f}]")
            for rect in noise_rects:
                rect.set_bounds(l_lo, m_lo, l_hi - l_lo, m_hi - m_lo)

        skies = [skyOrigb, skyLeshemb, skyNoLeshemb, skyCleanb, skyRestoredb]
        for im, ax, circ, sky, base in zip(ims, image_axes, src_circles, skies, base_titles):
            snr = measure_source_snr(sky, source_lm=sun_pos, src_radius=src_radius,
                                     noise_region=noise_region)
            fd = np.abs(sky).T
            im.set_data(fd)
            im.set_clim(vmin=np.nanmin(fd), vmax=np.nanmax(fd))
            circ.center = sun_pos
            ax.set_title(f"{base}\n{stokes}, SNR = {snr:.1f}")

        # Refresh sky source overlays on every image pane.
        t_mid = timeVec[tind[len(tind) // 2]]
        for artist in sky_overlay_artists:
            artist.remove()
        sky_overlay_artists.clear()
        for ax in image_axes:
            sky_overlay_artists.extend(vistools.overlay_sky_sources(ax, t_mid, NARRIBRI,
                                                                     scale=OVERLAY_SCALE))
        fig.canvas.draw()
        writer.grab_frame()

plt.close(fig)


In [ ]:
5000 * 17.8 / 1000

In [ ]:
orig_eval, orig_evec = np.linalg.eigh(calMean)
plt.plot(orig_eval)
